## CS421 Anomaly Detection 

#### Name: Justin Goh Kai Jun, Leong Zhe Cheng, Durga D/O Chandrasekaran
#### Group: G1T8

**Step 1: Imports and Setup**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import entropy, rankdata, spearmanr, pearsonr
from scipy.spatial.distance import cdist

from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.neighbors import LocalOutlierFactor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, roc_curve, precision_recall_curve
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import warnings, os
warnings.filterwarnings('ignore')

# random seed for clustering and autoencoder (deterministic)
np.random.seed(42)
torch.manual_seed(42)

# minimum AUC a model needs to enter the ensemble
ENSEMBLE_THRESHOLD = 0.65

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

**Step 2: Load Data**

In [ ]:
data = np.load('../training_batch_with_labels.npz')
X_raw = data['X']
y_raw = data['y']

XX = pd.DataFrame(X_raw, columns=['user', 'item', 'rating'])
yy = pd.DataFrame(y_raw, columns=['user', 'label'])
label_map = dict(zip(yy['user'], yy['label']))

print(f'Total interactions : {XX.shape[0]:,}')
print(f'Anomalous users    : {(yy["label"] == 1).sum()}')
print(f'Normal users       : {(yy["label"] == 0).sum()}')
print(f'Unique users       : {XX["user"].nunique()}')
print(f'Anomaly ratio      : {(yy["label"]==1).sum() / len(yy):.2%}')
print(f'Unique items       : {XX["item"].nunique()}')

In [ ]:
# ITERATIVE: load newly released test labels
LABEL_DIR = '../released_labels'

extra_files = sorted([f for f in os.listdir(LABEL_DIR) if f.startswith('released_week')])
extra_interactions = []
extra_labels       = []

for fname in extra_files:
    d  = np.load(os.path.join(LABEL_DIR, fname))
    df = pd.DataFrame(d['X'], columns=['user', 'item', 'rating'])
    
    # Remap user IDs to avoid collision with training IDs
    # Offset by a large number unique to this file
    offset = (extra_files.index(fname) + 1) * 10_000
    df['user'] = df['user'] + offset
    lbl = pd.DataFrame(d['y'], columns=['user', 'label'])
    lbl['user'] = lbl['user'] + offset
    extra_interactions.append(df)
    extra_labels.append(lbl)
    n_anom = (lbl['label']==1).sum()
    print(f'  Loaded {fname}: {len(df):,} interactions, {n_anom} anomalies')

if extra_interactions:
    XX_extra = pd.concat(extra_interactions, ignore_index=True)
    yy_extra = pd.concat(extra_labels, ignore_index=True)
    XX = pd.concat([XX, XX_extra], ignore_index=True)
    yy = pd.concat([yy, yy_extra], ignore_index=True)
    label_map.update(dict(zip(yy_extra['user'], yy_extra['label'])))
    print(f'\nAfter stacking: {XX.shape[0]:,} interactions, {(yy["label"]==1).sum()} anomalies')

else:
    print('No released label files found. Running on original training data only.')

In [ ]:
print(f'Total interactions : {XX.shape[0]:,}')
print(f'Anomalous users    : {(yy["label"] == 1).sum()}')
print(f'Normal users       : {(yy["label"] == 0).sum()}')
print(f'Unique users       : {XX["user"].nunique()}')
print(f'Anomaly ratio      : {(yy["label"]==1).sum() / len(yy):.2%}')
print(f'Unique items       : {XX["item"].nunique()}')

In [ ]:
print('Interactions (XX):'); display(XX.head())
print('Labels (yy):');       display(yy.head())

**Step 3: Feature Engineering**

**Helper function 1:** `compute_item_stats` calculates three things across all interactions: the mean rating per item, the number of ratings per item (popularity), and a percentile rank of each item's popularity from 0 to 1. These are needed for the attack-profile features.

In [ ]:
def compute_item_stats(interactions_df):
    """Compute item-level stats needed for attack-profile features."""
    item_mean   = interactions_df.groupby('item')['rating'].mean()
    item_count  = interactions_df.groupby('item')['rating'].count()
    item_pop_rank = item_count.rank(pct=True)  # 0=least popular, 1=most popular
    return item_mean, item_count, item_pop_rank

**Helper function 2:** `engineer_all_features` then loops over every user and builds a 38-number vector describing their behaviour. These break into three groups:
- **Rating statistics**: `mean`, `std`, `variance`, `min`, `max`, `range`, `fraction of each star level (0-5)`, `entropy of the rating distribution`, `fractions of extreme/zero/five/mid ratings`, `bimodality score`, and `skewness`. These capture the overall shape of how a user rates.
    
- **Item behaviour**: `number of interactions`, `unique items rated`, `density`, `fraction of repeat-rated items`, `item ID gap statistics` (mean, std, min gap between sorted item IDs), `sequential run fraction`, `item-rating autocorrelation` (whether ratings systematically change with item ID order), and `burst score` (whether 80% of interactions are concentrated on very few items)
  
- **Attack-profile features**: `avg_attack_score` (how closely a user rates items near their mean -> an average attacker tries to blend in by mimicking item averages), `bandwagon_score` (Pearson correlation between a user's ratings and item popularity -> bandwagon attackers inflate popular items), `random_attack_score` (normalised entropy -> random attackers produce a uniform distribution across all star levels), `segment_score` (combines popular item concentration with extreme ratings), `love_hate_score` (the fraction of 0 and 5 star ratings combined), `rating_deviation_std` (how consistently a user deviates from item averages)<br><br>
After the loop, the cosine deviation is computed -> each user's star-fraction vector is compared to the centroid of normal users using cosine similarity, and the distance from that centroid becomes a feature.

In [ ]:
ALL_FEATURE_NAMES = [
    'mean_r', 'std_r', 'variance_r', 'min_r', 'max_r', 'range_r',
    'star0', 'star1', 'star2', 'star3', 'star4', 'star5',
    'entropy',
    'frac_extreme', 'frac_zero', 'frac_five', 'frac_mid',
    'frac_45', 'frac_01', 'bimodal',
    'rating_skew',
    'n_inter', 'n_unique_items', 'density',
    'mean_gap', 'std_gap', 'min_gap',
    'sequential_run_frac',
    'item_autocorr',
    'burst_score',
    'n_repeat_items',
    'cosine_dev',
    # Attack profile features
    'avg_attack_score',
    'bandwagon_score',
    'random_attack_score',
    'popular_item_frac',
    'unpopular_item_frac',
    'segment_score',
    'love_hate_score',
    'rating_deviation_std',
    'gini_items',          
    'rating_zscore_mean', 
    'rating_zscore_std',  
    'entropy_zscore',   
    'n_inter_zscore',    
    'item_coverage',     
]

ATTACK_FEATURES = {
    'avg_attack_score', 'bandwagon_score', 'random_attack_score',
    'popular_item_frac', 'unpopular_item_frac', 'segment_score',
    'love_hate_score', 'rating_deviation_std'
}

In [ ]:
# ── engineer_all_features — replace the full function ─────────────────────
NORMAL_MEAN_R    = None
NORMAL_STD_R     = None
NORMAL_STD_STD   = None
NORMAL_ENT_MEAN  = None
NORMAL_ENT_STD   = None
NORMAL_NINT_MEAN = None
NORMAL_NINT_STD  = None

def engineer_all_features(interactions_df, n_items=1000, normal_centroid=None,
                           item_mean=None, item_count=None, item_pop_rank=None):
    """Build per-user feature matrix. Returns (feat_arr, user_ids, normal_centroid)."""
    global NORMAL_MEAN_R, NORMAL_STD_R, NORMAL_STD_STD
    global NORMAL_ENT_MEAN, NORMAL_ENT_STD, NORMAL_NINT_MEAN, NORMAL_NINT_STD

    if item_mean is None:
        item_mean, item_count, item_pop_rank = compute_item_stats(interactions_df)

    features = []
    user_ids = interactions_df['user'].unique()

    for uid in user_ids:
        u_df    = interactions_df[interactions_df['user'] == uid]
        ratings = u_df['rating'].values.astype(float)
        items   = u_df['item'].values

        # ── Basic rating stats ────────────────────────────────────────────
        mean_r     = ratings.mean()
        std_r      = ratings.std()    if len(ratings) > 1 else 0.0
        variance_r = ratings.var()    if len(ratings) > 1 else 0.0
        min_r, max_r = ratings.min(), ratings.max()
        range_r    = max_r - min_r

        star_fracs     = np.bincount(ratings.astype(int), minlength=6) / max(len(ratings), 1)
        rating_entropy = entropy(star_fracs + 1e-9)

        frac_extreme = np.mean((ratings == 0) | (ratings == 5))
        frac_zero    = np.mean(ratings == 0)
        frac_five    = np.mean(ratings == 5)
        frac_mid     = np.mean((ratings >= 2) & (ratings <= 3))
        frac_45      = np.mean(ratings >= 4)
        frac_01      = np.mean(ratings <= 1)
        bimodal      = frac_45 + frac_01
        rating_skew  = ((ratings - mean_r)**3).mean() / (std_r**3 + 1e-9)

        # ── Interaction / item features ───────────────────────────────────
        n_interactions = len(ratings)
        n_unique_items = len(np.unique(items))
        density        = n_interactions / n_items
        item_counts_u  = pd.Series(items).value_counts()
        n_repeat_items = (item_counts_u > 1).sum() / max(len(item_counts_u), 1)

        sorted_items = np.sort(items)
        item_gaps    = np.diff(sorted_items) if len(sorted_items) > 1 else np.array([0])
        mean_gap     = item_gaps.mean()
        std_gap      = item_gaps.std()  if len(item_gaps) > 1 else 0.0
        min_gap      = item_gaps.min()
        sequential_run_frac = (item_gaps == 1).sum() / max(len(item_gaps), 1)

        if len(items) > 3:
            sort_idx      = np.argsort(items)
            sorted_r      = ratings[sort_idx]
            item_autocorr, _ = spearmanr(np.arange(len(sorted_r)), sorted_r)
            item_autocorr = abs(item_autocorr) if not np.isnan(item_autocorr) else 0.0
        else:
            item_autocorr = 0.0

        if n_unique_items > 1:
            sorted_counts = np.sort(item_counts_u.values)[::-1]
            cumulative    = np.cumsum(sorted_counts) / sorted_counts.sum()
            items_for_80  = np.searchsorted(cumulative, 0.8) + 1
            burst_score   = 1.0 - (items_for_80 / n_unique_items)
        else:
            burst_score = 1.0

        cosine_dev = 0.0  # filled in after loop

        # ── Attack profile features ───────────────────────────────────────
        rated_items_means = item_mean.reindex(items).fillna(item_mean.mean()).values
        avg_attack_score  = 1.0 - np.mean(np.abs(ratings - rated_items_means)) / 5.0

        item_pops = item_pop_rank.reindex(items).fillna(0.5).values
        if len(items) > 3:
            corr, _ = pearsonr(item_pops, ratings)
            bandwagon_score = abs(corr) if not np.isnan(corr) else 0.0
        else:
            bandwagon_score = 0.0

        max_entropy         = entropy(np.ones(6) / 6)
        random_attack_score = rating_entropy / max_entropy

        popular_items       = set(item_pop_rank[item_pop_rank >= 0.8].index)
        unpopular_items     = set(item_pop_rank[item_pop_rank <= 0.2].index)
        popular_item_frac   = np.mean([i in popular_items   for i in items])
        unpopular_item_frac = np.mean([i in unpopular_items for i in items])
        segment_score       = popular_item_frac * frac_extreme
        love_hate_score     = frac_zero + frac_five
        rating_deviation_std = np.std(ratings - rated_items_means) if len(ratings) > 1 else 0.0

        # ── Attack-type-agnostic features ────────────────────────────

        # 1. Gini coefficient of item interactions
        if n_unique_items > 1:
            gc = np.sort(item_counts_u.values).astype(float)
            gc = gc / gc.sum()
            n  = len(gc)
            gini_items = (2 * np.sum(np.arange(1, n+1) * gc) - (n + 1)) / n
        else:
            gini_items = 1.0

        # 2-5. Z-scores relative to normal users
        if NORMAL_MEAN_R is not None:
            rating_zscore_mean = abs(mean_r - NORMAL_MEAN_R)     / (NORMAL_STD_R    + 1e-9)
            rating_zscore_std  = abs(std_r  - NORMAL_STD_STD[0]) / (NORMAL_STD_STD[1] + 1e-9)
            entropy_zscore     = abs(rating_entropy - NORMAL_ENT_MEAN)  / (NORMAL_ENT_STD  + 1e-9)
            n_inter_zscore     = abs(n_interactions - NORMAL_NINT_MEAN) / (NORMAL_NINT_STD + 1e-9)
        else:
            rating_zscore_mean = mean_r
            rating_zscore_std  = std_r
            entropy_zscore     = rating_entropy
            n_inter_zscore     = float(n_interactions)

        # 6. Item coverage 
        item_coverage = n_unique_items / n_items

        features.append(np.array([
            mean_r, std_r, variance_r, min_r, max_r, range_r,
            *star_fracs,
            rating_entropy,
            frac_extreme, frac_zero, frac_five, frac_mid,
            frac_45, frac_01, bimodal,
            rating_skew,
            n_interactions, n_unique_items, density,
            mean_gap, std_gap, min_gap,
            sequential_run_frac,
            item_autocorr,
            burst_score,
            n_repeat_items,
            cosine_dev,
            avg_attack_score,
            bandwagon_score,
            random_attack_score,
            popular_item_frac,
            unpopular_item_frac,
            segment_score,
            love_hate_score,
            rating_deviation_std,
            # new
            gini_items,
            rating_zscore_mean,
            rating_zscore_std,
            entropy_zscore,
            n_inter_zscore,
            item_coverage,
        ]))

    feat_arr  = np.array(features)
    star_cols = list(range(6, 12))

    if normal_centroid is None:
        normal_centroid = feat_arr[:, star_cols].mean(axis=0)

    star_vecs     = feat_arr[:, star_cols]
    norms         = np.linalg.norm(star_vecs, axis=1, keepdims=True) + 1e-9
    centroid_norm = np.linalg.norm(normal_centroid) + 1e-9
    cosine_sim    = (star_vecs / norms) @ (normal_centroid / centroid_norm)
    feat_arr[:, ALL_FEATURE_NAMES.index('cosine_dev')] = 1.0 - cosine_sim

    return feat_arr, user_ids, normal_centroid

In [ ]:
ITEM_MEAN, ITEM_COUNT, ITEM_POP_RANK = compute_item_stats(XX)

# Pass 1: raw values, no z-scores yet
feat_matrix_raw, feat_user_ids, _ = engineer_all_features(
    XX, item_mean=ITEM_MEAN, item_count=ITEM_COUNT, item_pop_rank=ITEM_POP_RANK
)
y_true = np.array([label_map[u] for u in feat_user_ids])
normal_mask = y_true == 0

# Compute normal centroid from star fracs
NORMAL_CENTROID = feat_matrix_raw[normal_mask, 6:12].mean(axis=0)

# Compute normal user stats for z-score features
_nm = feat_matrix_raw[normal_mask]
_mean_r_idx    = ALL_FEATURE_NAMES.index('mean_r')
_std_r_idx     = ALL_FEATURE_NAMES.index('std_r')
_ent_idx       = ALL_FEATURE_NAMES.index('entropy')
_nint_idx      = ALL_FEATURE_NAMES.index('n_inter')

NORMAL_MEAN_R    = _nm[:, _mean_r_idx].mean()
NORMAL_STD_R     = _nm[:, _mean_r_idx].std()
NORMAL_STD_STD   = (_nm[:, _std_r_idx].mean(), _nm[:, _std_r_idx].std())
NORMAL_ENT_MEAN  = _nm[:, _ent_idx].mean()
NORMAL_ENT_STD   = _nm[:, _ent_idx].std()
NORMAL_NINT_MEAN = _nm[:, _nint_idx].mean()
NORMAL_NINT_STD  = _nm[:, _nint_idx].std()

# Pass 2: now z-scores are real deviations from normal distribution
feat_matrix_all, feat_user_ids, NORMAL_CENTROID = engineer_all_features(
    XX, normal_centroid=NORMAL_CENTROID,
    item_mean=ITEM_MEAN, item_count=ITEM_COUNT, item_pop_rank=ITEM_POP_RANK
)
y_true = np.array([label_map[u] for u in feat_user_ids])

print(f'Feature matrix : {feat_matrix_all.shape}')
print(f'Total anomalies: {y_true.sum()} / {len(y_true)}')

**Step 4: Feature Selection**

Each feature is evaluated individually by computing its AUC against the true labels. Features below 0.55 AUC (worse than random after flipping) are dropped as they add noise without signal.

The notebook also identifies the top 3 features, and splits the remaining selected features into two views -> a rating view (features about how users rate) and an item view (features about which items users interact with). These views are used to train separate GMM models, giving the ensemble more diversity.

In [ ]:
feat_df = pd.DataFrame(feat_matrix_all, columns=ALL_FEATURE_NAMES)

feature_aucs = {}
for col in ALL_FEATURE_NAMES:
    auc = roc_auc_score(y_true, feat_df[col].values)
    feature_aucs[col] = auc if auc >= 0.5 else 1 - auc

auc_series = pd.Series(feature_aucs).sort_values(ascending=False)

KEEP_THRESHOLD = 0.55
colors = []
for f in auc_series.index:
    if auc_series[f] < KEEP_THRESHOLD:
        colors.append('#e74c3c')
    elif f in ATTACK_FEATURES:
        colors.append('#e67e22')
    else:
        colors.append('#2ecc71')

plt.figure(figsize=(7, 6))
auc_series.plot(kind='barh', color=colors)
plt.axvline(KEEP_THRESHOLD, color='black', linestyle='--', linewidth=1.2,
            label=f'Threshold ({KEEP_THRESHOLD})')
plt.xlabel('AUC')
plt.title('Per-feature AUC  (green=original, orange=attack profile, red=dropped)')
plt.legend()
plt.tight_layout()
plt.show()

SELECTED_FEATURES = [f for f in ALL_FEATURE_NAMES if feature_aucs[f] >= KEEP_THRESHOLD]
DROPPED_FEATURES  = [f for f in ALL_FEATURE_NAMES if feature_aucs[f] < KEEP_THRESHOLD]
TOP3              = [f for f, _ in sorted(feature_aucs.items(), key=lambda x: x[1], reverse=True)[:3]]
selected_idx = [ALL_FEATURE_NAMES.index(f) for f in SELECTED_FEATURES]
top3_idx     = [ALL_FEATURE_NAMES.index(f) for f in TOP3]

print(f'Selected ({len(SELECTED_FEATURES)}): {SELECTED_FEATURES}')
print(f'Dropped  ({len(DROPPED_FEATURES)}):  {DROPPED_FEATURES}')
print(f'Top-3    : {TOP3}')

# Feature views for diverse ensemble 
RATING_FEATURES = [f for f in SELECTED_FEATURES
                   if f in {'mean_r','std_r','variance_r','min_r','max_r','range_r',
                             'star0','star1','star2','star3','star4','star5',
                             'entropy','frac_extreme','frac_zero','frac_five',
                             'frac_mid','frac_45','frac_01','bimodal','rating_skew',
                             'cosine_dev','avg_attack_score','random_attack_score',
                             'love_hate_score','rating_deviation_std'}]
ITEM_FEATURES   = [f for f in SELECTED_FEATURES
                   if f in {'n_inter','n_unique_items','density','mean_gap','std_gap',
                             'min_gap','sequential_run_frac','item_autocorr','burst_score',
                             'n_repeat_items','bandwagon_score','popular_item_frac',
                             'unpopular_item_frac','segment_score'}]

rating_idx = [ALL_FEATURE_NAMES.index(f) for f in RATING_FEATURES]
item_idx   = [ALL_FEATURE_NAMES.index(f) for f in ITEM_FEATURES]

print(f'Rating view features ({len(RATING_FEATURES)}): {RATING_FEATURES}')
print(f'Item view features   ({len(ITEM_FEATURES)}):   {ITEM_FEATURES}')

**Step 5: Scaling normal only**

Four separate StandardScaler objects are fit (full feature set, top-3, the rating view, and the item view). Critically, all four are fit only on the rows where y_true == 0 (normal users). The scaler learns what "normal" looks like, and anything that deviates from that after scaling is naturally more visible.

In [ ]:
feat_matrix   = feat_matrix_all[:, selected_idx]
feat_top3     = feat_matrix_all[:, top3_idx]
feat_rating   = feat_matrix_all[:, rating_idx]
feat_item     = feat_matrix_all[:, item_idx]

normal_mask   = y_true == 0

scaler = StandardScaler()
scaler.fit(feat_matrix[normal_mask])
feat_scaled = scaler.transform(feat_matrix)
X_train_normal = feat_scaled[normal_mask]

top3_scaler = StandardScaler()
top3_scaler.fit(feat_top3[normal_mask])
feat_top3_scaled = top3_scaler.transform(feat_top3)

rating_scaler = StandardScaler()
rating_scaler.fit(feat_rating[normal_mask])
feat_rating_scaled = rating_scaler.transform(feat_rating)

item_scaler = StandardScaler()
item_scaler.fit(feat_item[normal_mask])
feat_item_scaled = item_scaler.transform(feat_item)

X_normal_rating = feat_rating_scaled[normal_mask]
X_normal_item   = feat_item_scaled[normal_mask]

print(f'Full feature matrix  : {feat_scaled.shape}')
print(f'Rating view          : {feat_rating_scaled.shape}')
print(f'Item view            : {feat_item_scaled.shape}')
print(f'Normal training rows : {X_train_normal.shape[0]}')

**Others: Utilities (Helper functions)**

**Helper function 1:** `evaluate` computes AUC, F1, precision and recall for any model. Tt finds the optimal threshold by maximising F1 on the precision-recall curve. 

In [ ]:
def evaluate(y_true_eval, scores, model_name):
    """
    Report AUC + metrics at TWO thresholds:
      • Best-F1  : maximises F1 (balanced, lower threshold)
      • High-Prec: highest threshold where recall >= 0.40 (flag fewer, more precisely)
    """
    auc = roc_auc_score(y_true_eval, scores)
    precisions, recalls, thresholds = precision_recall_curve(y_true_eval, scores)

    # Best-F1 threshold
    f1s         = 2 * precisions * recalls / (precisions + recalls + 1e-9)
    best_f1_idx = np.argmax(f1s)
    thr_f1      = thresholds[best_f1_idx] if best_f1_idx < len(thresholds) else 0.5

    # High-precision threshold: highest precision where recall >= 0.40
    valid = [(p, r, t) for p, r, t in zip(precisions[:-1], recalls[:-1], thresholds)
             if r >= 0.40]
    if valid:
        best_hp  = max(valid, key=lambda x: x[0])
        thr_prec = best_hp[2]
    else:
        thr_prec = thr_f1

    def metrics_at(thr):
        yp = (scores >= thr).astype(int)
        return (f1_score(y_true_eval, yp, zero_division=0),
                precision_score(y_true_eval, yp, zero_division=0),
                recall_score(y_true_eval, yp, zero_division=0),
                int(yp.sum()))

    f1_b, pr_b, re_b, n_b = metrics_at(thr_f1)
    f1_p, pr_p, re_p, n_p = metrics_at(thr_prec)

    print(f'[{model_name}]  AUC={auc:.4f}')
    print(f'  Best-F1   thr={thr_f1:.3f}  F1={f1_b:.3f}  Prec={pr_b:.3f}  Rec={re_b:.3f}  flags={n_b}')
    print(f'  High-Prec thr={thr_prec:.3f}  F1={f1_p:.3f}  Prec={pr_p:.3f}  Rec={re_p:.3f}  flags={n_p}')
    return {
        'model': model_name, 'AUC': auc,
        'F1': f1_b, 'Precision': pr_b, 'Recall': re_b,
        'F1_hp': f1_p, 'Precision_hp': pr_p, 'Recall_hp': re_p,
        'thr_f1': thr_f1, 'thr_prec': thr_prec,
    }


def find_high_prec_threshold(scores, y_true_eval, min_recall=0.40):
    """
    Return the score threshold that maximises precision while recall >= min_recall.
    Apply this to submitted scores to rank truly anomalous users at the top.
    """
    precisions, recalls, thresholds = precision_recall_curve(y_true_eval, scores)
    valid = [(p, r, t) for p, r, t in zip(precisions[:-1], recalls[:-1], thresholds)
             if r >= min_recall]
    if not valid:
        f1s = 2 * precisions * recalls / (precisions + recalls + 1e-9)
        return thresholds[np.argmax(f1s[:-1])] if len(thresholds) > 0 else 0.5
    return max(valid, key=lambda x: x[0])[2]

**Helper function 2:** `normalize` maps any array to [0,1] by min-max scaling.<br>
**Helper function 3:** `rank_normalize` replaces values with their percentile rank<br>
**Helper function 4:** `power_stretch` is the score stretching function that rank-normalises first (so all models are on a uniform scale), then raises to the power 0.4, which pushes high scores toward 1.0 and low scores toward 0.0<br>
**Helper function 5:**: `hbos_score` implements HBOS from scratch. For each feature, it builds a histogram on training data and scores test users by how unlikely their value is in each histogram bin, summing across all features.

In [ ]:
def normalize(arr):
    mn, mx = arr.min(), arr.max()
    return (arr - mn) / (mx - mn + 1e-9)

def rank_normalize(arr):
    return rankdata(arr) / len(arr)

def precision_stretch(arr, power=5.0):
    """
    Rank-normalise then apply power > 1 to sharpen separation.
    power=5 : very aggressive — true anomalies go to ~1.0, borderline to ~0.0.
    power=3 : moderate sharpening.
    Higher power = higher precision, lower recall.
    """
    return np.power(rank_normalize(arr), power)

def voting_consensus_score(score_dict, threshold_pct=0.85):
    """
    For each user, count how many models rank them in the TOP (1-threshold_pct)%.
    A user who is in the top 15% of EVERY model is almost certainly anomalous.
    Returns a consensus score [0..1] = fraction of models that flag this user.
    This is the primary tool for achieving near-100% precision.
    """
    votes = np.zeros(len(next(iter(score_dict.values()))))
    for name, scores in score_dict.items():
        # Flag = ranked in top (1-threshold_pct) of this model
        ranked = rank_normalize(scores)
        votes += (ranked >= threshold_pct).astype(float)
    return votes / len(score_dict)  # fraction of models that agree

def hbos_score(X_train, X_test, n_bins=10):
    scores = np.zeros(len(X_test))
    for j in range(X_train.shape[1]):
        hist, bin_edges = np.histogram(X_train[:, j], bins=n_bins, density=True)
        hist = np.maximum(hist, 1e-10)
        for i, val in enumerate(X_test[:, j]):
            bin_idx = np.searchsorted(bin_edges[1:], val, side='right')
            bin_idx = min(bin_idx, n_bins - 1)
            scores[i] += -np.log(hist[bin_idx] + 1e-10)
    return scores

method_scores = {} 
oof_scores    = {}  
results       = []

**Step 6: Unsupervised models**

**GMM Model:** Three GMM models are trained
- **1st:** on the full selected features
- **2nd:** one on the rating view
- **3rd:** one on the item view

For each, the number of components (2 to 6) is swept and the best is kept. All three are trained exclusively on X_train_normal (normal users only). The anomaly score is the negative log-likelihood (how unlikely a user's feature vector is under the normal distribution the GMM learned). A user who looks nothing like any of the normal clusters gets a high score.

In [ ]:
# GMM on full features
best_gmm_n, best_gmm_auc = 1, 0
for n in [2, 3, 4, 5, 6]:
    g = GaussianMixture(n_components=n, covariance_type='full', random_state=42)
    g.fit(X_train_normal)
    sc  = -g.score_samples(feat_scaled)
    auc = roc_auc_score(y_true, sc)
    if auc > best_gmm_auc:
        best_gmm_n, best_gmm_auc = n, auc

gmm = GaussianMixture(n_components=best_gmm_n, covariance_type='full', random_state=42)
gmm.fit(X_train_normal)
gmm_scores = -gmm.score_samples(feat_scaled)
auc_gmm    = roc_auc_score(y_true, gmm_scores)
method_scores['GMM'] = (normalize(gmm_scores), auc_gmm)
results.append(evaluate(y_true, gmm_scores, f'GMM n={best_gmm_n}'))

# GMM on rating view
best_rn, best_rauc, best_rcov = 2, 0, 'full'
for n in [2, 3, 4, 5]:
    for cov in ['full', 'diag', 'tied']:
        g = GaussianMixture(n_components=n, covariance_type=cov, random_state=42)
        g.fit(X_normal_rating)
        sc  = -g.score_samples(feat_rating_scaled)
        auc = roc_auc_score(y_true, sc)
        if auc > best_rauc:
            best_rn, best_rauc, best_rcov = n, auc, cov

gmm_rating = GaussianMixture(n_components=best_rn, covariance_type=best_rcov, random_state=42)
gmm_rating.fit(X_normal_rating)
gmm_rating_scores = -gmm_rating.score_samples(feat_rating_scaled)
auc_gmm_r = roc_auc_score(y_true, gmm_rating_scores)
method_scores['GMM (rating view)'] = (normalize(gmm_rating_scores), auc_gmm_r)
results.append(evaluate(y_true, gmm_rating_scores, f'GMM rating n={best_rn} cov={best_rcov}'))

# GMM on item view
best_in2, best_iauc, best_icov = 2, 0, 'full'
for n in [2, 3, 4, 5]:
    for cov in ['full', 'diag', 'tied']:
        g = GaussianMixture(n_components=n, covariance_type=cov, random_state=42)
        g.fit(X_normal_item)
        sc  = -g.score_samples(feat_item_scaled)
        auc = roc_auc_score(y_true, sc)
        if auc > best_iauc:
            best_in2, best_iauc, best_icov = n, auc, cov

gmm_item = GaussianMixture(n_components=best_in2, covariance_type=best_icov, random_state=42)
gmm_item.fit(X_normal_item)
gmm_item_scores = -gmm_item.score_samples(feat_item_scaled)
auc_gmm_i = roc_auc_score(y_true, gmm_item_scores)
method_scores['GMM (item view)'] = (normalize(gmm_item_scores), auc_gmm_i)
results.append(evaluate(y_true, gmm_item_scores, f'GMM item n={best_in2} cov={best_icov}'))
print(f'GMM sweep done. Best: full={best_gmm_n}, rating={best_rn}/{best_rcov}, item={best_in2}/{best_icov}')

**Isolation Forest**: Randomly splits the feature space -> anomalous users tend to get isolated in fewer splits, giving them a higher anomaly score. It is also trained on normal users only, with the contamination hyperparameter swept.<br>
**One-Class SVM:** Learns a tight boundary around the normal users in feature space. Anything outside that boundary scores as anomalous. The nu parameter (controlling how tight the boundary is) is swept.<br>
**LOF (Local Outlier Factor):** Compares each user's local density to its neighbours. A user in a sparse region surrounded by dense normal users gets a high LOF score. The number of neighbours k is swept.<br>
**HBOS:** Scores each user by how unusual they are in each feature dimension independently, then sums across features.

In [ ]:
# Isolation Forest 
best_cont, best_if_auc = 0.05, 0
for cont in [0.01, 0.05, 0.07, 0.09, 0.1, 0.12]:
    ifl = IsolationForest(n_estimators=400, contamination=cont, random_state=42)
    ifl.fit(X_train_normal)
    sc  = -ifl.decision_function(feat_scaled)
    auc = roc_auc_score(y_true, sc)
    if auc > best_if_auc:
        best_cont, best_if_auc = cont, auc

iforest = IsolationForest(n_estimators=400, contamination=best_cont, random_state=42)
iforest.fit(X_train_normal)
iforest_scores = -iforest.decision_function(feat_scaled)
auc_if = roc_auc_score(y_true, iforest_scores)
method_scores['Isolation Forest'] = (normalize(iforest_scores), auc_if)
results.append(evaluate(y_true, iforest_scores, f'Isolation Forest cont={best_cont}'))

# One-Class SVM 
best_nu, best_ocsvm_auc = 0.1, 0
for nu in [0.05, 0.08, 0.1, 0.12, 0.15, 0.2]:
    oc = OneClassSVM(kernel='rbf', nu=nu, gamma='scale')
    oc.fit(X_train_normal)
    sc  = -oc.decision_function(feat_scaled)
    auc = roc_auc_score(y_true, sc)
    if auc > best_ocsvm_auc:
        best_nu, best_ocsvm_auc = nu, auc

ocsvm = OneClassSVM(kernel='rbf', nu=best_nu, gamma='scale')
ocsvm.fit(X_train_normal)
ocsvm_scores = -ocsvm.decision_function(feat_scaled)
auc_ocsvm = roc_auc_score(y_true, ocsvm_scores)
method_scores['One-Class SVM'] = (normalize(ocsvm_scores), auc_ocsvm)
results.append(evaluate(y_true, ocsvm_scores, f'One-Class SVM nu={best_nu}'))

# LOF
best_lof_k, best_lof_auc = 5, 0
for k in [5, 10, 20, 30]:
    lof_tmp = LocalOutlierFactor(n_neighbors=k, novelty=True, contamination=0.09)
    lof_tmp.fit(X_train_normal)
    sc  = -lof_tmp.decision_function(feat_scaled)
    auc = roc_auc_score(y_true, sc)
    if auc > best_lof_auc:
        best_lof_k, best_lof_auc = k, auc

lof = LocalOutlierFactor(n_neighbors=best_lof_k, novelty=True, contamination=0.09)
lof.fit(X_train_normal)
lof_scores = -lof.decision_function(feat_scaled)
auc_lof = roc_auc_score(y_true, lof_scores)
method_scores['LOF'] = (normalize(lof_scores), auc_lof)
results.append(evaluate(y_true, lof_scores, f'LOF k={best_lof_k}'))

# HBOS
best_hbos_bins, best_hbos_auc = 10, 0
for nb in [5, 10, 15, 20, 30]:
    sc  = hbos_score(X_train_normal, feat_scaled, n_bins=nb)
    auc = roc_auc_score(y_true, sc)
    if auc > best_hbos_auc:
        best_hbos_bins, best_hbos_auc = nb, auc

hbos_scores = hbos_score(X_train_normal, feat_scaled, n_bins=best_hbos_bins)
auc_hbos = roc_auc_score(y_true, hbos_scores)
method_scores['HBOS'] = (normalize(hbos_scores), auc_hbos)
results.append(evaluate(y_true, hbos_scores, f'HBOS bins={best_hbos_bins}'))

**Step 7: Supervised models**

The class imbalance ratio between normal and anomalous users (~10:1) is computed and passed to all supervised models as scale_pos_weight or class_weight='balanced', which tells the models to treat each anomaly as worth 10 normal users in the loss function.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
n_neg = (y_true == 0).sum()
n_pos = (y_true == 1).sum()
imbalance_ratio = n_neg / n_pos
print(f'Class imbalance: {imbalance_ratio:.1f}:1')

In [ ]:
# Logistic Regression

lr_oof = np.zeros(len(y_true))
for fold, (tr_idx, vl_idx) in enumerate(skf.split(feat_scaled, y_true)):
    lr_f = LogisticRegression(class_weight='balanced', C=0.3, max_iter=1000, random_state=42)
    lr_f.fit(feat_scaled[tr_idx], y_true[tr_idx])
    lr_oof[vl_idx] = lr_f.predict_proba(feat_scaled[vl_idx])[:, 1]

auc_lr = roc_auc_score(y_true, lr_oof)
method_scores['Logistic Regression'] = (normalize(lr_oof), auc_lr)
oof_scores['Logistic Regression'] = lr_oof
results.append(evaluate(y_true, lr_oof, 'Logistic Regression (5-fold CV)'))

lr_full = LogisticRegression(class_weight=None, C=0.3, max_iter=1000, random_state=42)
lr_full.fit(feat_scaled, y_true)

**LightGBM** uses `num_leaves=15`, reg_alpha=0.1 and reg_lambda=1.0 for L1/L2 regularisation, and subsample=0.8 which means each tree only sees 80% of the training data.

In [ ]:
# LightGBM
lgbm_oof = np.zeros(len(y_true))
lgbm_fold_models = []
for fold, (tr_idx, vl_idx) in enumerate(skf.split(feat_scaled, y_true)):
    lgbm_f = LGBMClassifier(
        n_estimators=500, learning_rate=0.03, num_leaves=15,
        min_child_samples=10, reg_alpha=0.1, reg_lambda=1.0,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=imbalance_ratio,  # no upweighting — model must be confident before flagging
        random_state=42, verbose=-1
    )
    lgbm_f.fit(feat_scaled[tr_idx], y_true[tr_idx])
    lgbm_oof[vl_idx] = lgbm_f.predict_proba(feat_scaled[vl_idx])[:, 1]
    lgbm_fold_models.append(lgbm_f)

auc_lgbm = roc_auc_score(y_true, lgbm_oof)
method_scores['LightGBM'] = (normalize(lgbm_oof), auc_lgbm)
oof_scores['LightGBM'] = lgbm_oof
results.append(evaluate(y_true, lgbm_oof, 'LightGBM (5-fold CV)'))

lgbm_full = LGBMClassifier(
    n_estimators=200, learning_rate=0.03, num_leaves=7,
    min_child_samples=20, reg_alpha=0.1, reg_lambda=5.0,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=imbalance_ratio,
    random_state=42, verbose=-1
)
lgbm_full.fit(feat_scaled, y_true)

importances = pd.Series(lgbm_full.feature_importances_,
                         index=SELECTED_FEATURES).sort_values(ascending=True)
plt.figure(figsize=(8, 6))
importances.plot(kind='barh', color='steelblue')
plt.title('LightGBM Feature Importances')
plt.tight_layout()
plt.show()

**XGBoost** uses `max_depth=3` (shallower trees than default) and `min_child_weight=5` which requires at least 5 users to create a new leaf node, preventing the model from creating very specific rules for individual anomalies.

In [ ]:
# XGBoost
xgb_oof = np.zeros(len(y_true))
xgb_fold_models = []
for fold, (tr_idx, vl_idx) in enumerate(skf.split(feat_scaled, y_true)):
    xgb_f = XGBClassifier(
        n_estimators=200, learning_rate=0.03, max_depth=2,
        min_child_weight=10, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=1.0, reg_lambda=1.0,
        scale_pos_weight=1.0,  # precision-first: no anomaly upweighting
        use_label_encoder=False, eval_metric='logloss',
        random_state=42, verbosity=0
    )
    xgb_f.fit(feat_scaled[tr_idx], y_true[tr_idx])
    xgb_oof[vl_idx] = xgb_f.predict_proba(feat_scaled[vl_idx])[:, 1]
    xgb_fold_models.append(xgb_f)

auc_xgb = roc_auc_score(y_true, xgb_oof)
method_scores['XGBoost'] = (normalize(xgb_oof), auc_xgb)
oof_scores['XGBoost'] = xgb_oof
results.append(evaluate(y_true, xgb_oof, 'XGBoost (5-fold CV)'))

xgb_full = XGBClassifier(
    n_estimators=500, learning_rate=0.03, max_depth=3,
    min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    scale_pos_weight=imbalance_ratio,
    use_label_encoder=False, eval_metric='logloss',
    random_state=42, verbosity=0
)
xgb_full.fit(feat_scaled, y_true)

**Step 8: Stacking**

In [ ]:
# Build stacking OOF matrix
# Supervised OOF scores (CV-evaluated, unbiased)
stack_names = ['LightGBM', 'XGBoost', 'Logistic Regression']
stack_oof   = np.column_stack([oof_scores[n] for n in stack_names])

# Top unsupervised scores as additional meta-features
unsup_names  = ['GMM', 'Isolation Forest', 'LOF', 'HBOS']
unsup_stack  = np.column_stack([method_scores[n][0] for n in unsup_names])
stack_full   = np.hstack([stack_oof, unsup_stack])

print(f'Stacking meta-feature matrix: {stack_full.shape}')

# Meta-learner: LightGBM with scale_pos_weight=imbalance_ratio and strong regularisation.
meta_oof = np.zeros(len(y_true))
meta_models_stack = []
for fold, (tr_idx, vl_idx) in enumerate(skf.split(stack_full, y_true)):
    meta_lgbm = LGBMClassifier(
        n_estimators=100, learning_rate=0.05, num_leaves=7,
        min_child_samples=10,
        reg_alpha=2.0, reg_lambda=5.0,   
        scale_pos_weight=imbalance_ratio,            
        random_state=42, verbose=-1
    )
    meta_lgbm.fit(stack_full[tr_idx], y_true[tr_idx])
    meta_oof[vl_idx] = meta_lgbm.predict_proba(stack_full[vl_idx])[:, 1]
    meta_models_stack.append(meta_lgbm)

auc_meta = roc_auc_score(y_true, meta_oof)
method_scores['Stacking'] = (normalize(meta_oof), auc_meta)
oof_scores['Stacking'] = meta_oof
results.append(evaluate(y_true, meta_oof, 'Stacking meta-learner'))

# Final meta-model trained on full data 
meta_full = LGBMClassifier(
    n_estimators=100, learning_rate=0.05, num_leaves=7,
    min_child_samples=10, reg_alpha=2.0, reg_lambda=5.0,
    scale_pos_weight=imbalance_ratio, random_state=42, verbose=-1
)
meta_full.fit(stack_full, y_true)
print(f'Stacking AUC: {auc_meta:.4f}')

# ── Voting consensus on training data (for threshold calibration) ─────────
# Build dict of all OOF scores for voting
oof_vote_dict = {
    'LightGBM'  : normalize(lgbm_oof),
    'XGBoost'   : normalize(xgb_oof),
    'LR'        : normalize(lr_oof),
    'Stacking'  : normalize(meta_oof),
    'GMM'       : method_scores['GMM'][0],
    'IF'        : method_scores['Isolation Forest'][0],
    'LOF'       : method_scores['LOF'][0],
    'HBOS'      : method_scores['HBOS'][0],
    'DAE'       : method_scores.get('DAE', method_scores['GMM'])[0],
    'VAE'       : method_scores.get('VAE', method_scores['GMM'])[0],
}

**Step 9: Deep Learning (Denoising AE + VAE)**

In [ ]:
class DenoisingAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, latent_dim=16, noise_std=0.15):
        super().__init__()
        self.noise_std = noise_std
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.LeakyReLU(0.1),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, latent_dim), nn.LeakyReLU(0.1),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.LeakyReLU(0.1),
            nn.Linear(hidden_dim, input_dim)
        )

    def forward(self, x, add_noise=True):
        if add_noise and self.training:
            x = x + torch.randn_like(x) * self.noise_std
        return self.decoder(self.encoder(x))

    def reconstruction_error(self, x):
        self.eval()
        with torch.no_grad():
            return ((x - self.forward(x, add_noise=False))**2).mean(dim=1).cpu().numpy()

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, latent_dim=16):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.LeakyReLU(0.1))
        self.fc_mu     = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        self.decoder   = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.LeakyReLU(0.1),
            nn.Linear(hidden_dim, input_dim)
        )

    def encode(self, x):
        h = self.enc(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        return self.decoder(z), mu, logvar

In [ ]:
def train_dae(model, X_n_t, X_all_t, epochs=150, lr=1e-3):
    opt   = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit  = nn.MSELoss()
    loader = DataLoader(TensorDataset(X_n_t), batch_size=64, shuffle=True)
    model.train()
    for epoch in range(epochs):
        for (batch,) in loader:
            opt.zero_grad()
            crit(model(batch, True), batch).backward()
            opt.step()
        sched.step()
        if (epoch+1) % 50 == 0:
            model.eval()
            with torch.no_grad():
                loss = crit(model(X_n_t, False), X_n_t).item()
            print(f'  DAE epoch {epoch+1}  loss={loss:.6f}')
            model.train()
    return model.reconstruction_error(X_all_t)


def train_vae(model, X_n_t, X_all_t, epochs=150, lr=1e-3):
    opt   = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loader = DataLoader(TensorDataset(X_n_t), batch_size=64, shuffle=True)
    model.train()
    for epoch in range(epochs):
        for (batch,) in loader:
            opt.zero_grad()
            recon, mu, logvar = model(batch)
            loss = (nn.functional.mse_loss(recon, batch, reduction='sum')
                    - 0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()))
            loss.backward()
            opt.step()
        sched.step()
        if (epoch+1) % 50 == 0:
            print(f'  VAE epoch {epoch+1}')
    model.eval()
    scores = []
    with torch.no_grad():
        for i in range(len(X_all_t)):
            x = X_all_t[i].unsqueeze(0)
            recon, mu, logvar = model(x)
            rec = nn.functional.mse_loss(recon, x, reduction='sum').item()
            kl  = (-0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())).item()
            scores.append(rec + 0.5 * kl)
    return np.array(scores)

In [ ]:
input_dim  = feat_scaled.shape[1]
X_n_t      = torch.FloatTensor(X_train_normal).to(DEVICE)
X_all_t    = torch.FloatTensor(feat_scaled).to(DEVICE)

# Sweep latent dim
best_latent, best_ae_auc = 16, 0
print('DAE latent sweep:')
for ldim in [8, 12, 16, 24]:
    tmp = DenoisingAutoencoder(input_dim, 64, ldim, 0.15).to(DEVICE)
    sc  = train_dae(tmp, X_n_t, X_all_t, epochs=80)
    auc = roc_auc_score(y_true, sc)
    print(f'  latent={ldim}  AUC={auc:.4f}')
    if auc > best_ae_auc:
        best_latent, best_ae_auc = ldim, auc

dae_model  = DenoisingAutoencoder(input_dim, 64, best_latent, 0.15).to(DEVICE)
dae_scores = train_dae(dae_model, X_n_t, X_all_t, epochs=150)
auc_dae    = roc_auc_score(y_true, dae_scores)
method_scores['DAE'] = (normalize(dae_scores), auc_dae)
results.append(evaluate(y_true, dae_scores, f'DAE latent={best_latent}'))

vae_model  = VAE(input_dim, 64, best_latent).to(DEVICE)
vae_scores = train_vae(vae_model, X_n_t, X_all_t, epochs=150)
auc_vae    = roc_auc_score(y_true, vae_scores)
method_scores['VAE'] = (normalize(vae_scores), auc_vae)
results.append(evaluate(y_true, vae_scores, f'VAE latent={best_latent}'))
print(f'DL done: DAE={auc_dae:.4f}, VAE={auc_vae:.4f}')

**Step 10: Adaptive Ensemble + Score Stretching**

**Score stretching** is applied to the final submission scores.
Rank-normalise -> power transform with `power=0.4` spreads scores to fully use [0,1].
This directly addresses the compressed [0.03, 0.75] range problem.

In [ ]:
SUPERVISED_NAMES  = {'LightGBM', 'XGBoost', 'Logistic Regression', 'Stacking'}
CV_AUC_THRESHOLD  = 0.80
SUPERVISED_PENALTY = 0.5
MAX_SUPERVISED_WEIGHT = 0.20  # cap each supervised model to prevent overfitted models dominating

print(f'Method AUCs (threshold={ENSEMBLE_THRESHOLD}):')
for name, (_, auc) in sorted(method_scores.items(), key=lambda x: -x[1][1]):
    flag = 'In ' if auc >= ENSEMBLE_THRESHOLD else 'Out'
    print(f'  {name:30s}  AUC={auc:.4f}  {flag}')

good   = {n: (s, auc) for n, (s, auc) in method_scores.items() if auc >= ENSEMBLE_THRESHOLD}
excess = {}
for n, (_, auc) in good.items():
    ex = auc - ENSEMBLE_THRESHOLD
    if n in SUPERVISED_NAMES and auc < CV_AUC_THRESHOLD:
        ex *= SUPERVISED_PENALTY
        print(f'  [Adaptive] {n} penalised (AUC={auc:.3f} < {CV_AUC_THRESHOLD})')
    excess[n] = ex

total_excess = sum(excess.values())
weights_ens  = {n: ex / total_excess for n, ex in excess.items()}

# Cap supervised models and redistribute surplus to unsupervised
surplus = 0.0
for n in SUPERVISED_NAMES:
    if n in weights_ens and weights_ens[n] > MAX_SUPERVISED_WEIGHT:
        surplus += weights_ens[n] - MAX_SUPERVISED_WEIGHT
        weights_ens[n] = MAX_SUPERVISED_WEIGHT

# Distribute surplus proportionally among unsupervised models only
unsup_in_ensemble = [n for n in weights_ens if n not in SUPERVISED_NAMES]
if unsup_in_ensemble and surplus > 0:
    unsup_total = sum(weights_ens[n] for n in unsup_in_ensemble)
    for n in unsup_in_ensemble:
        weights_ens[n] += surplus * (weights_ens[n] / unsup_total)

# Renormalise to ensure weights sum to 1
total_w = sum(weights_ens.values())
weights_ens = {n: w / total_w for n, w in weights_ens.items()}

print('\nEnsemble weights:')
for name, w in sorted(weights_ens.items(), key=lambda x: -x[1]):
    print(f'  {name:30s}  weight={w:.3f}')

ensemble_scores = sum(weights_ens[n] * good[n][0] for n in weights_ens)
auc_ens = roc_auc_score(y_true, ensemble_scores)

best_single_name = max(method_scores, key=lambda x: method_scores[x][1])
best_single_auc  = method_scores[best_single_name][1]

submit_scores = ensemble_scores if auc_ens >= best_single_auc else method_scores[best_single_name][0]
submit_label  = 'Adaptive Ensemble' if auc_ens >= best_single_auc else f'Best: {best_single_name}'
submit_auc    = max(auc_ens, best_single_auc)

print(f'\nRaw ensemble AUC : {auc_ens:.4f}')
print(f'Best single AUC  : {best_single_name} ({best_single_auc:.4f})')
print(f'Submit           : {submit_label} ({submit_auc:.4f})')

results.append(evaluate(y_true, ensemble_scores, 'Adaptive Ensemble'))

# ── Precision threshold calibration on training data ─────────────────────
PREC_THRESHOLD_META = find_high_prec_threshold(meta_oof, y_true, min_recall=0.20)
PREC_THRESHOLD_ENS  = find_high_prec_threshold(ensemble_scores, y_true, min_recall=0.20)

def show_precision_recall_tradeoff(scores, y_true_eval, name):
    precs, recs, thrs = precision_recall_curve(y_true_eval, scores)
    print(f'\n{name} — Precision/Recall tradeoff:')
    print(f'  {"Threshold":>10}  {"Precision":>10}  {"Recall":>8}  {"Flags":>6}')
    for min_rec in [0.10, 0.20, 0.30, 0.40, 0.50]:
        valid = [(p, r, t) for p, r, t in zip(precs[:-1], recs[:-1], thrs) if r >= min_rec]
        if valid:
            p, r, t = max(valid, key=lambda x: x[0])
            n_flags = int((scores >= t).sum())
            print(f'  {t:>10.4f}  {p:>10.4f}  {r:>8.4f}  {n_flags:>6}')

show_precision_recall_tradeoff(meta_oof,        y_true, 'Stacking meta')
show_precision_recall_tradeoff(ensemble_scores, y_true, 'Adaptive ensemble')
print(f'\nThreshold chosen for submission (recall≥20%):')
print(f'  Stacking : {PREC_THRESHOLD_META:.4f}')
print(f'  Ensemble : {PREC_THRESHOLD_ENS:.4f}')

**Step 11: Visualisations**

In [ ]:
# ROC curves
plt.figure(figsize=(10, 8))
for name, (scores, auc) in sorted(method_scores.items(), key=lambda x: -x[1][1]):
    fpr, tpr, _ = roc_curve(y_true, scores)
    lw    = 1.5 if auc >= ENSEMBLE_THRESHOLD else 0.8
    style = '-'  if auc >= ENSEMBLE_THRESHOLD else '--'
    plt.plot(fpr, tpr, linestyle=style, linewidth=lw, label=f'{name} ({auc:.3f})')
fpr_e, tpr_e, _ = roc_curve(y_true, submit_scores)
plt.plot(fpr_e, tpr_e, 'k-', linewidth=2.5, label=f'{submit_label} ({submit_auc:.3f})')
plt.plot([0,1],[0,1],'grey',linestyle=':',alpha=0.4)
plt.xlabel('FPR'); plt.ylabel('TPR')
plt.title('ROC Curves — All Methods')
plt.legend(fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
# Score distributions
n_methods = len(method_scores)
ncols = 3; nrows = (n_methods + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4*nrows))
axes = axes.flatten()
for i, (name, (scores, auc)) in enumerate(method_scores.items()):
    ax = axes[i]
    ax.hist(scores[y_true==0], bins=30, alpha=0.6, label='Normal', color='steelblue', density=True)
    ax.hist(scores[y_true==1], bins=30, alpha=0.6, label='Anomalous', color='crimson', density=True)
    ax.set_title(f'{name}\nAUC={auc:.3f}', fontsize=9); ax.legend(fontsize=7)
for j in range(n_methods, len(axes)): axes[j].set_visible(False)
plt.suptitle('Score Distributions', fontsize=12)
plt.tight_layout()
plt.show()

**Step 12: Ablation**

In [ ]:
ablation = {n: auc for n, (_, auc) in method_scores.items()}
ablation['Final'] = submit_auc
abl_df = pd.DataFrame.from_dict(ablation, orient='index', columns=['AUC']).sort_values('AUC')

attack_names = {'GMM (rating view)', 'GMM (item view)', 'Stacking'}
colors_abl = []
for i in abl_df.index:
    if 'Final' in i: colors_abl.append('crimson')
    elif i in attack_names: colors_abl.append('darkorange')
    elif i in SUPERVISED_NAMES: colors_abl.append('#f39c12')
    else: colors_abl.append('steelblue')

plt.figure(figsize=(12, 8))
abl_df['AUC'].plot(kind='barh', color=colors_abl)
plt.axvline(0.5, color='grey', linestyle='--', alpha=0.5)
plt.axvline(ENSEMBLE_THRESHOLD, color='steelblue', linestyle='--', alpha=0.5)
plt.xlabel('AUC'); plt.title('Ablation')
plt.tight_layout(); plt.show()
print(abl_df.sort_values('AUC', ascending=False).to_string())

**Step 13: Results table**

In [ ]:
results_df = pd.DataFrame(results)
# Show both F1-threshold and high-precision-threshold metrics
cols_show = ['model','AUC','F1','Precision','Recall','F1_hp','Precision_hp','Recall_hp']
cols_show = [c for c in cols_show if c in results_df.columns]
results_df = results_df[cols_show].sort_values('AUC', ascending=False).reset_index(drop=True)
print('=== Final Model Comparison ===')
print('Columns: AUC | Best-F1 threshold metrics | High-Precision threshold metrics')
print(results_df.to_string(index=False, float_format='%.4f'))

**Step 14: Submissions**

In [ ]:
TEST_FILE = '../second_batch.npz'   # <- update each week

data_test = np.load(TEST_FILE)
df_test   = pd.DataFrame(data_test['X'], columns=['user', 'item', 'rating'])

# Use same item stats and centroid from training
test_feat_all, test_user_ids, _ = engineer_all_features(
    df_test, n_items=1000,
    normal_centroid=NORMAL_CENTROID,
    item_mean=ITEM_MEAN, item_count=ITEM_COUNT, item_pop_rank=ITEM_POP_RANK
)

sort_order    = np.argsort(test_user_ids)
test_user_ids = test_user_ids[sort_order]
test_feat_all = test_feat_all[sort_order]

test_feat        = test_feat_all[:, selected_idx]
test_top3        = test_feat_all[:, top3_idx]
test_rating      = test_feat_all[:, rating_idx]
test_item        = test_feat_all[:, item_idx]
test_scaled      = scaler.transform(test_feat)
test_top3_s      = top3_scaler.transform(test_top3)
test_rating_s    = rating_scaler.transform(test_rating)
test_item_s      = item_scaler.transform(test_item)
X_test_t         = torch.FloatTensor(test_scaled).to(DEVICE)

print(f'Test users        : {len(test_user_ids)}')
print(f'User ID range     : [{test_user_ids.min()}, {test_user_ids.max()}]')
print(f'Test interactions : {len(df_test):,}')

In [ ]:
def vae_scores_fn(model, X_t):
    model.eval()
    scores = []
    with torch.no_grad():
        for i in range(len(X_t)):
            x = X_t[i].unsqueeze(0)
            recon, mu, logvar = model(x)
            rec = nn.functional.mse_loss(recon, x, reduction='sum').item()
            kl  = (-0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())).item()
            scores.append(rec + 0.5 * kl)
    return np.array(scores)

# Score all models
t_gmm        = normalize(-gmm.score_samples(test_scaled))
t_gmm_rating = normalize(-gmm_rating.score_samples(test_rating_s))
t_gmm_item   = normalize(-gmm_item.score_samples(test_item_s))
t_ocsvm      = normalize(-ocsvm.decision_function(test_scaled))
t_iforest    = normalize(-iforest.decision_function(test_scaled))
t_lof        = normalize(-lof.decision_function(test_scaled))
t_hbos       = normalize(hbos_score(X_train_normal, test_scaled, n_bins=best_hbos_bins))
t_lr         = normalize(lr_full.predict_proba(test_scaled)[:, 1])
t_lgbm       = normalize(np.mean([m.predict_proba(test_scaled)[:,1] 
                             for m in lgbm_fold_models], axis=0))
t_xgb        = normalize(np.mean([m.predict_proba(test_scaled)[:,1] 
                             for m in xgb_fold_models], axis=0))
t_dae        = normalize(dae_model.reconstruction_error(X_test_t))
t_vae        = normalize(vae_scores_fn(vae_model, X_test_t))

# Stacking meta prediction
t_stack_sup  = np.column_stack([t_lgbm, t_xgb, t_lr])
t_stack_unsup = np.column_stack([t_gmm, t_iforest, t_lof, t_hbos])
t_stack_full = np.hstack([t_stack_sup, t_stack_unsup])
t_meta       = normalize(np.mean([m.predict_proba(t_stack_full)[:,1] 
                             for m in meta_models_stack], axis=0))

print('All test scores computed.')
print(f'Score ranges before stretching:')
for name, sc in [('GMM', t_gmm), ('DAE', t_dae), ('LGBM', t_lgbm), ('Meta', t_meta)]:
    print(f'  {name}: [{sc.min():.3f}, {sc.max():.3f}]')

In [ ]:
# With this:
unsup_test = (
    0.30 * normalize(t_gmm) +
    0.25 * normalize(t_iforest) +
    0.20 * normalize(t_hbos) +
    0.15 * normalize(t_dae) +
    0.10 * normalize(t_ocsvm)
)
combined = 0.40 * normalize(t_meta) + 0.60 * unsup_test
combined = normalize(combined)

combined = 0.75 * normalize(t_meta) + 0.25 * vote_scores_test
combined = normalize(combined)

# # ── Layer 3: Precision sharpening ────────────────────────────────────────
np.savez('../submission.npz', predictions=sub_final)

# ── Evaluate on training data ─────────────────────────────────────────────
unsup_train = (
    0.30 * method_scores['GMM'][0] +
    0.25 * method_scores['Isolation Forest'][0] +
    0.20 * method_scores['HBOS'][0] +
    0.15 * method_scores['DAE'][0] +
    0.10 * method_scores['One-Class SVM'][0]
)
combined_train = 0.40 * normalize(meta_oof) + 0.60 * unsup_train
combined_train = normalize(combined_train)
sub_train = precision_stretch(combined_train, power=5.0)

print('\n=== Submission Score Analysis (training data) ===')
print(f'Score range: [{sub_final.min():.4f}, {sub_final.max():.4f}]')
print(f'Predictions: {len(sub_final)}')
print()
precisions, recalls, thresholds = precision_recall_curve(y_true, sub_train)
print(f'{"Threshold":>10}  {"Precision":>10}  {"Recall":>8}  {"Flags":>6}')
for thr in [0.95, 0.90, 0.85, 0.80, 0.70, 0.60, 0.50]:
    n_flags = int((sub_train >= thr).sum())
    tp      = int(((sub_train >= thr) & (y_true == 1)).sum())
    prec_v  = tp / max(n_flags, 1)
    rec_v   = tp / max(y_true.sum(), 1)
    print(f'  {thr:>8.2f}  {prec_v:>10.4f}  {rec_v:>8.4f}  {n_flags:>6}')

print()
print('AUC of final submission scores (training):')
print(f'  {roc_auc_score(y_true, sub_train):.4f}')
print()
print('  → zip submission.npz → submission.zip → upload to Codabench')
print(f'  User ID range: [{test_user_ids.min()}, {test_user_ids.max()}]')
print(f'  {len(sub_final)} predictions sorted by ascending user_id')

In [ ]:
# # ── Single Submission: Maximum Precision ─────────────────────────────────────
# #
# # STRATEGY: Only flag users that MULTIPLE independent models unanimously agree on.
# # This is the only reliable way to achieve near-100% precision.
# #
# # Three-layer approach:
# #   Layer 1 — Voting consensus: count how many models rank this user in top 10%
# #   Layer 2 — Stacking meta score: LightGBM learns when models agree
# #   Layer 3 — Precision sharpening (power=5): creates bimodal distribution
# #
# # Result: a score near 1.0 means "all models strongly agree this user is anomalous"
# #         a score near 0.0 means "at most 1 or 2 models flag this user"

# # ── Build test voting dict ────────────────────────────────────────────────
# test_vote_dict = {
#     'LightGBM'  : t_lgbm,
#     'XGBoost'   : t_xgb,
#     'LR'        : t_lr,
#     'Stacking'  : t_meta,
#     'GMM'       : t_gmm,
#     'IF'        : t_iforest,
#     'LOF'       : t_lof,
#     'HBOS'      : t_hbos,
#     'DAE'       : t_dae,
#     'VAE'       : t_vae,
# }

# # ── Layer 1: Voting score ─────────────────────────────────────────────────
# # threshold_pct=0.90 means "flag only users in top 10% of each model"
# # Using top 10% threshold: a user flagged by 10/10 models is almost certainly anomalous
# vote_scores_test = voting_consensus_score(test_vote_dict, threshold_pct=0.85)
# vote_scores_train = voting_consensus_score(oof_vote_dict, threshold_pct=0.90)

# # Show what voting achieves on training data
# print('Voting consensus on training data:')
# for min_vote in [1.0, 0.9, 0.8, 0.7, 0.5]:
#     flagged = (vote_scores_train >= min_vote).sum()
#     tp      = ((vote_scores_train >= min_vote) & (y_true == 1)).sum()
#     prec    = tp / max(flagged, 1)
#     rec     = tp / max(y_true.sum(), 1)
#     print(f'  votes≥{min_vote:.0%}:  {flagged:4d} flagged  TP={tp:3d}  Prec={prec:.3f}  Rec={rec:.3f}')

# # ── Layer 2: Combine voting + stacking meta ───────────────────────────────
# # Weighted combination: stacking meta gets 60%, voting gets 40%
# # Both agree → score near 1.0. Only one agrees → score stays low.
# combined = 0.75 * normalize(t_meta) + 0.25 * vote_scores_test
# combined = normalize(combined)

# # ── Layer 3: Precision sharpening ────────────────────────────────────────
# # power=5: creates very bimodal distribution
# # Users in top 5% → score ~0.95+
# # Users in bottom 50% → score ~0.03 or less
# sub_final = precision_stretch(combined, power=3.0)

# np.savez('../submission.npz', predictions=sub_final)

# # ── Evaluate on training data ─────────────────────────────────────────────
# combined_train = 0.60 * normalize(meta_oof) + 0.40 * vote_scores_train
# combined_train = normalize(combined_train)
# sub_train = precision_stretch(combined_train, power=5.0)

# print('\n=== Submission Score Analysis (training data) ===')
# print(f'Score range: [{sub_final.min():.4f}, {sub_final.max():.4f}]')
# print(f'Predictions: {len(sub_final)}')
# print()
# precisions, recalls, thresholds = precision_recall_curve(y_true, sub_train)
# print(f'{"Threshold":>10}  {"Precision":>10}  {"Recall":>8}  {"Flags":>6}')
# for thr in [0.95, 0.90, 0.85, 0.80, 0.70, 0.60, 0.50]:
#     n_flags = int((sub_train >= thr).sum())
#     tp      = int(((sub_train >= thr) & (y_true == 1)).sum())
#     prec_v  = tp / max(n_flags, 1)
#     rec_v   = tp / max(y_true.sum(), 1)
#     print(f'  {thr:>8.2f}  {prec_v:>10.4f}  {rec_v:>8.4f}  {n_flags:>6}')

# print()
# print('AUC of final submission scores (training):')
# print(f'  {roc_auc_score(y_true, sub_train):.4f}')
# print()
# print('  → zip submission.npz → submission.zip → upload to Codabench')
# print(f'  User ID range: [{test_user_ids.min()}, {test_user_ids.max()}]')
# print(f'  {len(sub_final)} predictions sorted by ascending user_id')

In [ ]:
# ── Threshold Tuning Guide ────────────────────────────────────────────────────
# The submission uses CONTINUOUS scores (not binary labels).
# Codabench computes AUC from your continuous scores.
# Precision/Recall metrics are computed at a threshold Codabench chooses internally.
#
# The voting_consensus threshold_pct and precision_stretch power are the two
# main levers. Adjust them here and re-run Cell 57 to regenerate the submission.
#
# GUIDE:
#   threshold_pct=0.90, power=5  → ~100% precision, ~15-25% recall  (current)
#   threshold_pct=0.85, power=4  → ~90% precision,  ~30-40% recall
#   threshold_pct=0.80, power=3  → ~70% precision,  ~50-60% recall
#
# Since precision is the primary metric, keep threshold_pct high (0.85-0.95).
# Only lower it if you find the test anomalies look very different from training.

print('Voting threshold sensitivity (training data):')
print(f'  {"threshold_pct":>14}  {"power":>6}  {"Prec@top5%":>12}  {"Rec@top5%":>10}')
for thr_pct in [0.80, 0.85, 0.90, 0.95]:
    for pwr in [3.0, 5.0]:
        v_train = voting_consensus_score(oof_vote_dict, threshold_pct=thr_pct)
        c_train = normalize(0.60 * normalize(meta_oof) + 0.40 * v_train)
        s_train = precision_stretch(c_train, power=pwr)
        # Check top 5% of scores
        cutoff  = np.percentile(s_train, 95)
        flagged = (s_train >= cutoff).sum()
        tp      = ((s_train >= cutoff) & (y_true == 1)).sum()
        prec_v  = tp / max(flagged, 1)
        rec_v   = tp / max(y_true.sum(), 1)
        print(f'  {thr_pct:>14.0%}  {pwr:>6.1f}  {prec_v:>12.4f}  {rec_v:>10.4f}')

print()
print('Choose the row with the highest precision and acceptable recall.')
print('Update threshold_pct and power in Cell 57, then re-run to regenerate submission.')

In [ ]:
# ── Final Precision Summary ───────────────────────────────────────────────────
# Shows achievable precision for each base model at recall levels 10%-50%.
# Use this to understand which models are most trustworthy for precision.

print('Max achievable precision per model at various recall levels (training):')
print(f'{"Model":<25}  {"P@Rec≥10%":>10}  {"P@Rec≥20%":>10}  {"P@Rec≥30%":>10}  {"P@Rec≥50%":>10}')
print('-' * 72)

analysis = {
    'Stacking (meta)' : meta_oof,
    'LightGBM'        : lgbm_oof,
    'XGBoost'         : xgb_oof,
    'Logistic Reg'    : lr_oof,
    'GMM'             : gmm_scores,
    'Isolation Forest': iforest_scores,
    'LOF'             : lof_scores,
    'HBOS'            : hbos_scores,
    'DAE'             : dae_scores,
    'VAE'             : vae_scores,
    'Voting Consensus': vote_scores_train,
    'Final Combined'  : sub_train,
}

for name, scores in analysis.items():
    row = f'{name:<25}'
    for min_rec in [0.10, 0.20, 0.30, 0.50]:
        precs, recs, thrs = precision_recall_curve(y_true, scores)
        valid = [(p,) for p, r in zip(precs[:-1], recs[:-1]) if r >= min_rec]
        if valid:
            row += f'  {max(valid, key=lambda x: x[0])[0]:>10.4f}'
        else:
            row += f'  {"—":>10}'
    print(row)

print()
print('"Final Combined" = the actual submission scores.')
print('Target: P@Rec≥20% as close to 1.0 as possible.')